# Bachelor Thesis

© 2026 Yvan Richard   
University of St. Gallen, Spring Term 2026

## Reproduction of Barber et al. (2022)

---

## Foreword

In this notebook, the purpose is to reproduce Table I in Barber et al. (2022). The table depicts the summary statistics of the main data set.

## 1. Libraries & Data

I first load the relevant libraries and data.

In [1]:
# libs 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
import seaborn as sns
from datetime import datetime

In [8]:
# data
main = pd.read_csv("../../../data/processed/attention_sample.csv")

# parses dates
main["date"] = pd.to_datetime(main["date"], format="%Y-%m-%d")

/var/folders/7v/_v_y1jpx0rl056gg5rkjsw4r0000gn/T/ipykernel_44632/2291011228.py:2: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  main = pd.read_csv("../../../data/processed/attention_sample.csv")


In [9]:
# unique tickers list
unique_tickers = set(main["ticker"].unique())
print(f"Number of unique tickers: {len(unique_tickers)}")

Number of unique tickers: 1979


Once the data are loaded, I build the relevant variables for computing the summary statistics.

In [10]:
# positive price
main['prc'] = main['prc'].abs()

# size
main['size'] = main['shrout'] * main['prc'].abs()

# build binary variable 'has_users' indicating whether there are any users for that ticker on that day
main["has_users"] = (main["users_close"] > 0).astype(int)

I now compute Panel A

In [13]:
# summary stats Panel A
summary_stats_panel_a = main[["users_close", "users_last", "userchg", "userratio", "prc", "size", "ret",
                                "daily_buys", "daily_sells", "net_buys", "taq_retimb", "asvi"]].describe().T
print(summary_stats_panel_a)

                 count          mean           std          min  \
users_close  1037118.0  3.794270e+03  2.394779e+04      1.00000   
users_last   1037118.0  3.797049e+03  2.396766e+04      0.00000   
userchg      1015432.0  1.701591e+01  3.344176e+02 -17015.00000   
userratio    1015406.0  1.005207e+00  2.685814e-01      0.00632   
prc          1037118.0  6.682608e+01  1.515383e+02      0.08500   
size         1037095.0  1.451438e+07  5.537350e+07   1172.00000   
ret          1037095.0  2.852141e-04  3.697781e-02     -1.00000   
daily_buys   1031902.0  3.368872e+02  1.504895e+03      0.00000   
daily_sells  1031902.0  3.087040e+02  1.217371e+03      0.00000   
net_buys     1031902.0  2.818324e+01  4.531688e+02 -10131.00000   
taq_retimb   1031902.0  1.593261e-03  2.356671e-01     -1.00000   
asvi          967061.0  4.533398e-01  7.164092e-01      0.00000   

                       25%           50%           75%           max  
users_close      96.000000  3.060000e+02  1.193000e+03  9

In [14]:
# Panel B: Daily Observations (Summed Variable Averaged across Days)
# keep only observations where has_users == 1
main = main[main["has_users"] == 1]

# group by date and sum the relevant variables: [users_close, users_last, userchg, daily_buys, daily_sells, net_buys]
daily_summary = main.groupby("date").agg({
    "users_close": "sum",
    "users_last": "sum",
    "userchg": "sum",
    "daily_buys": "sum",
    "daily_sells": "sum",
    "net_buys": "sum",
    "has_users": "sum"
}).reset_index()

# summary stats Panel B
summary_stats_panel_b = daily_summary[["users_close", "users_last", "userchg", "daily_buys", "daily_sells", "net_buys", "has_users"]].describe().T
print(summary_stats_panel_b)

             count          mean           std      min         25%  \
users_close  554.0  7.103079e+06  4.798597e+06   3912.0  4267395.50   
users_last   554.0  7.108282e+06  4.804237e+06   3913.0  4269376.50   
userchg      554.0  3.118862e+04  6.320885e+04 -64713.0     3976.00   
daily_buys   554.0  6.274993e+05  3.587967e+05    346.0   416361.00   
daily_sells  554.0  5.750041e+05  2.806785e+05    330.0   407619.75   
net_buys     554.0  5.249521e+04  9.113193e+04 -54640.0     3082.00   
has_users    554.0  1.872054e+03  1.162032e+02     11.0     1863.00   

                   50%         75%         max  
users_close  5416782.0  6426848.50  20315536.0  
users_last   5417464.0  6429413.25  20324981.0  
userchg         8583.0    19657.75    467899.0  
daily_buys    464679.0   619317.75   2126338.0  
daily_sells   455302.5   598055.50   1728908.0  
net_buys       16926.0    49383.50    579263.0  
has_users       1878.0     1896.00      1917.0  


In [22]:
# check most popular tickers
top_tickers = main.groupby("ticker")["users_close"].sum().sort_values(ascending=False).head(10)
print(top_tickers)

ticker
ACB     211168422.0
F       192038116.0
GE      182523030.0
AAPL    136557927.0
MSFT    134402176.0
GPRO    125694624.0
FIT     113681676.0
DIS     110533627.0
AMD      93902001.0
CRON     93409958.0
Name: users_close, dtype: float64
